In [2]:
import sys
!{sys.executable} -m pip install transformers torch

ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_internal\cli\base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_internal\cli\req_command.py", line 67, in wrapper

   ---------------------------------------- 10.0/10.0 MB 48.1 MB/s eta 0:00:00
   ---------------------------------------- 2.4/2.4 MB 19.5 MB/s eta 0:00:00



    return func(self, options, args)
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_internal\commands\install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_internal\metadata\pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_vendor\packaging\version.py", line 56, in parse
    return Version(version)
  File "c:\Users\cutet\anaconda3\lib\site-packages\pip\_vendor\packaging\version.py", line 202, in __init__
    raise InvalidVersion(f"Invalid version: {version!r}")
pip._vendor.packaging.version.InvalidVersion: Invalid version: '4.0.0-unsupported'


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

print("Downloading MedCPT Article Encoder...")
tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")
print("Done")

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\cutet\anaconda3\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\cutet\.cache\huggingface\hub\models--ncbi--MedCPT-Article-Encoder. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Done


In [1]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import hashlib

# ── Load data ──────────────────────────────────────────────
radiomics_df = pd.read_excel(r"C:\Users\cutet\Desktop\ClinIQ\radiomic_features_normalized.xlsx", index_col=0)
notes_df = pd.read_excel(r"C:\Users\cutet\Desktop\ClinIQ\synthesized_clinical_notes.xlsx")

print(f"Radiomic cases:  {len(radiomics_df)}")
print(f"Clinical notes:  {len(notes_df)}")

# ── Find common case IDs ────────────────────────────────────
common_ids = set(radiomics_df.index) & set(notes_df["case_id"])
print(f"Common case IDs: {len(common_ids)}")

# ── Load MedCPT Article Encoder ─────────────────────────────
print("\nLoading MedCPT...")
tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")
model.eval()
print("MedCPT ready")

def embed_clinical_note(text):
    # MedCPT Article Encoder expects [title, abstract] pairs
    # We pass the note as both title and abstract
    encoded = tokenizer(
        [[text, text]],
        truncation=True,
        padding=True,
        return_tensors="pt",
        max_length=512
    )
    with torch.no_grad():
        output = model(**encoded)
    # CLS token as the representation
    return output.last_hidden_state[:, 0, :].squeeze().tolist()

# ── Connect to Qdrant ───────────────────────────────────────
client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "cliniq_cases"
RADIOMICS_DIM   = radiomics_df.shape[1]   # 54
CLINICAL_DIM    = 768                      # MedCPT output

# ── Create collection ───────────────────────────────────────
# Delete if exists so we start fresh
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "radiomics": VectorParams(size=RADIOMICS_DIM, distance=Distance.COSINE),
        "clinical":  VectorParams(size=CLINICAL_DIM,  distance=Distance.COSINE)
    }
)
print(f"Created collection: {COLLECTION_NAME}")
print(f"  radiomics vector dim: {RADIOMICS_DIM}")
print(f"  clinical vector dim:  {CLINICAL_DIM}")

# ── Ingest all cases ────────────────────────────────────────
notes_lookup = notes_df.set_index("case_id")["synthesized_note"].to_dict()

points = []
failed = []

for case_id in sorted(common_ids):
    try:
        # Radiomic vector — already normalized
        radiomic_vector = radiomics_df.loc[case_id].tolist()

        # Clinical vector — embed now
        note = notes_lookup[case_id]
        clinical_vector = embed_clinical_note(note)

        points.append(PointStruct(
            id=int(hashlib.sha256(case_id.encode()).hexdigest(), 16) % (10**15),  # Qdrant needs integer ID
            vector={
                "radiomics": radiomic_vector,
                "clinical":  clinical_vector
            },
            payload={
                "case_id":  case_id,
                "note":     note
            }
        ))

        print(f"  ✓ {case_id}")

    except Exception as e:
        print(f"  ✗ {case_id} — {e}")
        failed.append(case_id)

# ── Upload to Qdrant in batches ─────────────────────────────
BATCH_SIZE = 10
for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i+BATCH_SIZE]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"Uploaded batch {i//BATCH_SIZE + 1}/{(len(points) + BATCH_SIZE - 1)//BATCH_SIZE}")

print(f"\nDone.")
print(f"  Stored:  {len(points)}")
print(f"  Failed:  {len(failed)}")
if failed:
    print(f"  Failed cases: {failed}")

# ── Verify ──────────────────────────────────────────────────
info = client.get_collection(COLLECTION_NAME)
print(f"\nQdrant collection info:")
print(f"  Points count: {info.points_count}")

Radiomic cases:  184
Clinical notes:  184
Common case IDs: 184

Loading MedCPT...
MedCPT ready
Created collection: cliniq_cases
  radiomics vector dim: 54
  clinical vector dim:  768
  ✓ UCSD-PTGBM-0002_01
  ✓ UCSD-PTGBM-0005_01
  ✓ UCSD-PTGBM-0005_02
  ✓ UCSD-PTGBM-0006_01
  ✓ UCSD-PTGBM-0007_01
  ✓ UCSD-PTGBM-0007_02
  ✓ UCSD-PTGBM-0008_01
  ✓ UCSD-PTGBM-0008_02
  ✓ UCSD-PTGBM-0008_03
  ✓ UCSD-PTGBM-0009_01
  ✓ UCSD-PTGBM-0010_01
  ✓ UCSD-PTGBM-0011_01
  ✓ UCSD-PTGBM-0012_01
  ✓ UCSD-PTGBM-0016_01
  ✓ UCSD-PTGBM-0017_01
  ✓ UCSD-PTGBM-0017_02
  ✓ UCSD-PTGBM-0018_01
  ✓ UCSD-PTGBM-0018_02
  ✓ UCSD-PTGBM-0019_01
  ✓ UCSD-PTGBM-0019_02
  ✓ UCSD-PTGBM-0020_01
  ✓ UCSD-PTGBM-0021_01
  ✓ UCSD-PTGBM-0022_01
  ✓ UCSD-PTGBM-0022_02
  ✓ UCSD-PTGBM-0023_01
  ✓ UCSD-PTGBM-0024_01
  ✓ UCSD-PTGBM-0025_01
  ✓ UCSD-PTGBM-0025_02
  ✓ UCSD-PTGBM-0025_03
  ✓ UCSD-PTGBM-0026_01
  ✓ UCSD-PTGBM-0029_01
  ✓ UCSD-PTGBM-0031_01
  ✓ UCSD-PTGBM-0032_01
  ✓ UCSD-PTGBM-0032_02
  ✓ UCSD-PTGBM-0032_03
  ✓ UCSD-PTG